# Table X — Most notable individual per polity

20 famous polities, temporally dispersed from the 3rd millennium BCE to the 20th century CE.

**Source**: `data/humans_clean.duckdb` (read-only).

**Polity assignment**: `individuals_cliopatria` (Cliopatria polygon + temporal floruit match).

**Notability metric**: `individuals.notability_general` — the project's composite notability index, defined as the geometric mean of the western and non-western indices (each = Wikipedia editions in that bucket + metadata completeness + external identifiers from issuers in that bucket). Geometric mean rewards individuals with balanced cross-cultural reach and penalises one-sided fame. Computed by `scripts/_one_off/add_notability_indices.py` and `scripts/_one_off/redefine_notability_general_geomean.py`.

**Polity dates**: pulled from `polities_periods_cliopatria` (min `from_year`, max `to_year`) — never invented.

Output is a TSV printed at the end of the notebook (copy-paste manually).

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

DB = Path('..') / 'data' / 'humans_clean.duckdb'
con = duckdb.connect(str(DB), read_only=True)
print('Connected to', DB.resolve())

Connected to /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura_database/data/humans_clean.duckdb


In [2]:
# 20 famous polities, ordered chronologically.
# polity_id is the primary key in polities_cliopatria; polity names can
# collide so we always join on the integer ID.
POLITIES = [
    (6,    'Akkadian Empire'),
    (23,   'New Kingdom of Egypt'),
    (91,   'Achaemenid Empire'),
    (65,   'Macedonian Empire'),
    (93,   'Roman Republic'),
    (114,  'Ptolemaic Kingdom'),
    (204,  'Roman Empire'),
    (223,  'Sasanian Empire'),
    (366,  'Tang Dynasty'),
    (402,  'Carolingian Empire'),
    (414,  'Abbasid Caliphate'),
    (373,  'Byzantine Empire'),
    (729,  'Mongol Empire'),
    (397,  'Republic of Venice'),
    (523,  'Holy Roman Empire'),
    (896,  'Ming Dynasty'),
    (839,  'Ottoman Empire'),
    (1050, 'Tokugawa Shogunate'),
    (1117, 'Russian Empire'),
    (1102, 'Kingdom of Great Britain'),
]
len(POLITIES)

20

In [3]:
# Polity periods straight from the database — never invented.
polity_periods = con.execute(
    """
    SELECT polity_id,
           MIN(from_year) AS year_start,
           MAX(to_year)   AS year_end
    FROM polities_periods_cliopatria
    WHERE polity_id = ANY(?)
    GROUP BY polity_id
    """,
    [[pid for pid, _ in POLITIES]],
).fetchdf().set_index('polity_id')
polity_periods

,year_start,year_end
polity_id,,
6,-2300,-2101
23,-1500,-801
65,-675,-292
91,-550,-327
93,-500,-32
114,-331,-28
204,-31,394
223,215,643
366,623,910


In [4]:
# For each polity, take the individual with the highest notability_general.
# individuals_cliopatria.polity_id is a ';'-joined string of polity IDs (one
# individual can be matched to several polities), so we split + list_contains.
def most_notable(polity_id: int):
    return con.execute(
        """
        SELECT i.wikidata_id,
               i.name_en,
               i.description_en,
               i.birthdate,
               i.deathdate,
               i.notability_general,
               i.notability_western,
               i.notability_non_western,
               i.wikimedia_links_count,
               ic.floruit_year
        FROM individuals_cliopatria ic
        JOIN individuals i USING(wikidata_id)
        WHERE list_contains(string_split(ic.polity_id, ';'), ?)
        ORDER BY i.notability_general DESC NULLS LAST
        LIMIT 1
        """,
        [str(polity_id)],
    ).fetchone()

rows = []
for pid, pname in POLITIES:
    r = most_notable(pid)
    period = polity_periods.loc[pid]
    rows.append({
        'polity_id': pid,
        'polity': pname,
        'year_start': int(period['year_start']),
        'year_end':   int(period['year_end']),
        'wikidata_id': r[0],
        'most_notable': r[1],
        'description': r[2],
        'birthdate': r[3],
        'deathdate': r[4],
        'notability_general':    r[5],
        'notability_western':    r[6],
        'notability_non_western':r[7],
        'wikipedia_languages':   r[8],
        'floruit_year': r[9],
    })

df = pd.DataFrame(rows).sort_values('year_start').reset_index(drop=True)
df

,polity_id,polity,year_start,year_end,wikidata_id,most_notable,description,birthdate,deathdate,notability_general,notability_western,notability_non_western,wikipedia_languages,floruit_year
0,6,Akkadian Empire,-2300,-2101,Q199461,Sargon of Akkad,founder of Akkadian Empire,-2350-01-01,-2300-01-01,45.497253,46,45,78,-2301
1,23,New Kingdom of Egypt,-1500,-801,Q12154,TutanKhamun,14th century BCE (18th dynasty) Egyptian pharaoh,-1340-01-01,-1323-01-01,72.828566,78,68,114,-1323
2,65,Macedonian Empire,-675,-292,Q8409,Alexander the Great,king of Macedonia and conqueror of Achaemenid ...,-0355-07-15,-0322-06-05,140.477756,138,143,259,-324
3,91,Achaemenid Empire,-550,-327,Q41155,Heraclitus,Greek philosopher (late 6th/early 5th-century BC),-0534-01-01,-0470-01-01,75.266194,103,55,144,-492
4,93,Roman Republic,-500,-32,Q1048,Julius Caesar,Roman general and dictator,-0099-07-01,-0043-03-13,150.598805,168,135,280,-57
5,114,Ptolemaic Kingdom,-331,-28,Q635,Cleopatra,queen of the Ptolemaic Kingdom of Egypt from 5...,-0068-01-11,-0029-08-10,100.349390,106,95,170,-34
6,204,Roman Empire,-31,394,Q302,Jesus Christ,central figure of Christianity (6 or 4 BC – AD...,-0005-01-01,0033-04-01,169.699735,154,187,341,29
7,223,Sasanian Empire,215,643,Q8467,Umar ibn Al-Khattāb,2nd Rashidun Caliph from 634 to 644,0586-01-01,0644-11-06,75.299402,63,90,150,628
8,366,Tang Dynasty,623,910,Q7071,Li Bai,Classical Chinese poet of the Tang dynasty (70...,0701-05-23,0762-12-04,103.489130,102,105,191,743
9,373,Byzantine Empire,633,1474,Q83100,Osman I,founder of the Ottoman Empire (died 1323/4),1258-08-30,1326-08-09,54.221767,49,60,103,1300


## Helpers — period label & life-dates label

Display formatting only — values themselves come from the DB.

In [5]:
def fmt_year(y: int) -> str:
    if y is None:
        return ''
    return f'{abs(int(y))} BCE' if int(y) < 0 else f'{int(y)} CE'

def fmt_period(y0, y1) -> str:
    return f'{fmt_year(y0)} – {fmt_year(y1)}'

def parse_year(iso_date: str):
    if iso_date is None or iso_date == '' or str(iso_date).startswith('_:'):
        return None
    s = str(iso_date)
    sign = -1 if s.startswith('-') else 1
    body = s[1:] if s.startswith('-') else s
    try:
        return sign * int(body.split('-')[0])
    except Exception:
        return None

def fmt_lifespan(birth, death):
    by, dy = parse_year(birth), parse_year(death)
    if by is None and dy is None:
        return ''
    return f'{fmt_year(by) if by is not None else "?"} – {fmt_year(dy) if dy is not None else "?"}'

df['period']   = [fmt_period(s, e) for s, e in zip(df['year_start'], df['year_end'])]
df['lifespan'] = [fmt_lifespan(b, d) for b, d in zip(df['birthdate'], df['deathdate'])]
df[['polity', 'period', 'most_notable', 'lifespan', 'description',
    'notability_general', 'notability_western', 'notability_non_western',
    'wikipedia_languages']]

,polity,period,most_notable,lifespan,description,notability_general,notability_western,notability_non_western,wikipedia_languages
0,Akkadian Empire,2300 BCE – 2101 BCE,Sargon of Akkad,2350 BCE – 2300 BCE,founder of Akkadian Empire,45.497253,46,45,78
1,New Kingdom of Egypt,1500 BCE – 801 BCE,TutanKhamun,1340 BCE – 1323 BCE,14th century BCE (18th dynasty) Egyptian pharaoh,72.828566,78,68,114
2,Macedonian Empire,675 BCE – 292 BCE,Alexander the Great,355 BCE – 322 BCE,king of Macedonia and conqueror of Achaemenid ...,140.477756,138,143,259
3,Achaemenid Empire,550 BCE – 327 BCE,Heraclitus,534 BCE – 470 BCE,Greek philosopher (late 6th/early 5th-century BC),75.266194,103,55,144
4,Roman Republic,500 BCE – 32 BCE,Julius Caesar,99 BCE – 43 BCE,Roman general and dictator,150.598805,168,135,280
5,Ptolemaic Kingdom,331 BCE – 28 BCE,Cleopatra,68 BCE – 29 BCE,queen of the Ptolemaic Kingdom of Egypt from 5...,100.349390,106,95,170
6,Roman Empire,31 BCE – 394 CE,Jesus Christ,5 BCE – 33 CE,central figure of Christianity (6 or 4 BC – AD...,169.699735,154,187,341
7,Sasanian Empire,215 CE – 643 CE,Umar ibn Al-Khattāb,586 CE – 644 CE,2nd Rashidun Caliph from 634 to 644,75.299402,63,90,150
8,Tang Dynasty,623 CE – 910 CE,Li Bai,701 CE – 762 CE,Classical Chinese poet of the Tang dynasty (70...,103.489130,102,105,191
9,Byzantine Empire,633 CE – 1474 CE,Osman I,1258 CE – 1326 CE,founder of the Ottoman Empire (died 1323/4),54.221767,49,60,103


## Table X — TSV (copy-paste)

Columns:

- **Polity** — name as it appears in `polities_cliopatria`.
- **Period** — `min(from_year) – max(to_year)` from `polities_periods_cliopatria`.
- **Most notable individual** — `argmax(notability_general)` over individuals matched to the polity in `individuals_cliopatria`.
- **Lifespan** — birth – death year from `individuals.birthdate` / `deathdate`.
- **Notability (general / W / NW)** — `notability_general`, `notability_western`, `notability_non_western` from `individuals`.
- **Wikipedia languages** — `wikimedia_links_count` (kept for reference).

In [6]:
tsv = df[[
    'polity',
    'period',
    'most_notable',
    'lifespan',
    'description',
    'notability_general',
    'notability_western',
    'notability_non_western',
    'wikipedia_languages',
]].rename(columns={
    'polity': 'Polity',
    'period': 'Period',
    'most_notable': 'Most notable individual',
    'lifespan': 'Lifespan',
    'description': 'Description',
    'notability_general':     'Notability (general)',
    'notability_western':     'Notability (W)',
    'notability_non_western': 'Notability (NW)',
    'wikipedia_languages':    'Wikipedia languages',
}).copy()
tsv['Notability (general)'] = tsv['Notability (general)'].round(1)
print(tsv.to_csv(sep='\t', index=False))

Polity	Period	Most notable individual	Lifespan	Description	Notability (general)	Notability (W)	Notability (NW)	Wikipedia languages
Akkadian Empire	2300 BCE – 2101 BCE	Sargon of Akkad	2350 BCE – 2300 BCE	founder of Akkadian Empire	45.5	46	45	78
New Kingdom of Egypt	1500 BCE – 801 BCE	TutanKhamun	1340 BCE – 1323 BCE	14th century BCE (18th dynasty) Egyptian pharaoh	72.8	78	68	114
Macedonian Empire	675 BCE – 292 BCE	Alexander the Great	355 BCE – 322 BCE	king of Macedonia and conqueror of Achaemenid Persia (356–323 BC)	140.5	138	143	259
Achaemenid Empire	550 BCE – 327 BCE	Heraclitus	534 BCE – 470 BCE	Greek philosopher (late 6th/early 5th-century BC)	75.3	103	55	144
Roman Republic	500 BCE – 32 BCE	Julius Caesar	99 BCE – 43 BCE	Roman general and dictator	150.6	168	135	280
Ptolemaic Kingdom	331 BCE – 28 BCE	Cleopatra	68 BCE – 29 BCE	queen of the Ptolemaic Kingdom of Egypt from 51 to 30 BCE	100.3	106	95	170
Roman Empire	31 BCE – 394 CE	Jesus Christ	5 BCE – 33 CE	central figure of Christianity (